In [6]:
# ==========================================
# 1. 必要なライブラリのインストール
# ==========================================
!pip install langchain-chroma langchain-ollama langchain-core -q

# ==========================================d
# 2. Google Colab上でOllamaを確実にインストール＆起動
# ==========================================
import subprocess
import time
import os
import socket
import tempfile # For temporary log files

print("Ollamaをインストール中...")

# Ollamaインストールパスを設定 (スクリプトが/usr/localにインストールするため、これを反映)
ollama_install_dir = "/usr/local/bin"
ollama_executable_path = f"{ollama_install_dir}/ollama"

# 依存関係zstdをインストール（Ollamaの依存関係）
print("依存関係zstdをインストール中...")
subprocess.run(
    ["sudo", "apt-get", "update", "-qq"], # Update package list silently
    check=True,
    capture_output=True
)
subprocess.run(
    ["sudo", "apt-get", "install", "-y", "zstd"],
    check=True,
    capture_output=True
)
print("zstdのインストールに成功しました。")

# Ollamaインストールスクリプトをダウンロード
install_script_path = "/tmp/install_ollama.sh"
print(f"Ollamaインストールスクリプトをダウンロード中: https://ollama.com/install.sh")
subprocess.run(
    ["curl", "-fsSL", "https://ollama.com/install.sh", "-o", install_script_path],
    check=True,
    capture_output=True
)
print(f"Ollamaインストールスクリプトを {install_script_path} にダウンロードしました。")

# Ollamaインストールスクリプトを実行
print("Ollamaインストールスクリプトを実行中... (これには数分かかる場合があります)")
# OLLAMA_INSTALL_DIR を指定しても、スクリプトが /usr/local にインストールする場合があるため、
# Python側ではインストール後のパスを /usr/local/bin と仮定する。
# スクリプトにOLLAMA_INSTALL_DIRを渡しても無視される可能性があるため、ここでは明示的に渡さない。
install_process = subprocess.run(
    ["bash", install_script_path],
    check=True,
    capture_output=True
    # env=install_env # スクリプトが無視するため、ここでは渡さない
)
print(f"インストールスクリプトの標準出力:\n{install_process.stdout.decode()}")
if install_process.stderr:
    print(f"インストールスクリプトの標準エラー:\n{install_process.stderr.decode()}")

# PATHを更新してollamaコマンドが実行可能になるようにする
# 現在のPythonプロセスと将来のsubprocessのためにPATHを更新
if ollama_install_dir not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = f"{ollama_install_dir}{os.pathsep}{os.environ.get('PATH', '')}"
print(f"Ollamaのインストールが完了しました。実行パス: {ollama_executable_path}")

# Ollamaサーバーをバックグラウンドで起動
print("Ollamaサーバーを起動中...")
if not os.path.exists(ollama_executable_path):
    raise FileNotFoundError(f"Ollama実行ファイルが見つかりません: {ollama_executable_path}. インストールに失敗した可能性があります。")

server_env = os.environ.copy() # Use the updated os.environ

# Redirect stdout and stderr to temporary files for debugging
stdout_log = tempfile.TemporaryFile(mode='w+')
stderr_log = tempfile.TemporaryFile(mode='w+')

try:
    # We've confirmed 'serve --help' doesn't list --embeddings. The error must be due to the model.
    # So, we'll remove the help check and directly try to start the server.

    server_process = subprocess.Popen(
        [ollama_executable_path, "serve"],
        stdout=stdout_log, # Redirect stdout to log file
        stderr=stderr_log, # Redirect stderr to log file
        env=server_env
    )

    # Ollamaサーバーが起動するまで待機する
    ollama_api_host = "127.0.0.1"
    ollama_api_port = 11434
    max_retries = 30
    for i in range(max_retries):
        try:
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.settimeout(1) # 1 second timeout
            s.connect((ollama_api_host, ollama_api_port))
            s.close()
            print("Ollamaサーバーが起動しました。")
            break
        except (socket.timeout, ConnectionRefusedError):
            print(f"Ollamaサーバーの起動を待機中... ({i+1}/{max_retries})")
            time.sleep(2) # Wait 2 seconds between retries
    else:
        # If the loop completes without breaking (server didn't start)
        print("-------------------------------------------------------------------")
        print("Ollamaサーバーが指定時間内に起動しませんでした。以下にログを表示します。")
        # Check if the server process has terminated and retrieve logs
        if server_process.poll() is not None:
            print("Ollamaサーバーが予期せず終了しました。ログを確認してください。")
            stdout_log.seek(0)
            stderr_log.seek(0)
            stdout_content = stdout_log.read()
            stderr_content = stderr_log.read()
            if stdout_content: print(f"Ollamaサーバーの標準出力:\n{stdout_content}")
            if stderr_content: print(f"Ollamaサーバーの標準エラー:\n{stderr_content}")
        else:
            print("Ollamaサーバーはまだ実行中ですが、ポートに応答していません。")
        print("-------------------------------------------------------------------")
        raise RuntimeError("Ollamaサーバーが指定時間内に起動しませんでした。")
finally:
    # Close temporary files
    stdout_log.close()
    stderr_log.close()

# 正しいOllamaのモデル名（チャット用）
chat_model_name = "lucas2024/llama-3-elyza-jp-8b:q5_k_m"
print(f"Ollamaから {chat_model_name} をダウンロード中... (約1〜2分)")

# モデルのプルが完了するまでしっかり待つ
pull_process = subprocess.run(
    [ollama_executable_path, "pull", chat_model_name],
    check=True, # Raise an exception if the command returns a non-zero exit code
    capture_output=True,
    env=server_env # Ensure updated PATH is used for `ollama pull`
)
print(f"モデルダウンロードの標準出力:\n{pull_process.stdout.decode()}")
if pull_process.stderr:
    print(f"モデルダウンロードの標準エラー:\n{pull_process.stderr.decode()}")

# 埋め込み用のモデル名
embedding_model_name = "nomic-embed-text"
print(f"Ollamaから埋め込み用モデル {embedding_model_name} をダウンロード中...")
pull_embedding_process = subprocess.run(
    [ollama_executable_path, "pull", embedding_model_name],
    check=True,
    capture_output=True,
    env=server_env
)
print(f"埋め込みモデルダウンロードの標準出力:\n{pull_embedding_process.stdout.decode()}")
if pull_embedding_process.stderr:
    print(f"埋め込みモデルダウンロードの標準エラー:\n{pull_embedding_process.stderr.decode()}")

print("OllamaとELYZAおよび埋め込みモデルの準備が完了しました！\n" + "="*40 + "\n")

# ==========================================
# 3. LangChain RAGパイプラインの構築と実行
# ==========================================
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def run_colab_rag():
    # 日本語のサンプルデータ
    sample_data = [
        Document(
            page_content="プロジェクト・アルファの内部コードネームは『シルバースァーファー』で、2026年10月にリリース予定です。",
            metadata={"source": "project_alpha.txt"}
        ),
        Document(
            page_content="プロジェクト・アルファの予算は120万ドルで、すべてマーケティングチームが管理しています。",
            metadata={"source": "finance_2026.txt"}
        )
    ]

    # テキストをベクトルに変換（OllamaがGPUを使って高速処理します）
    # ここでは、OllamaEmbeddingsがOllamaサーバーにアクセスできるように、
    # 環境変数 OLLAMA_HOST を設定する必要があるかもしれません。
    # ただし、デフォルトでは 'http://localhost:11434' を使用するため、
    # サーバーが起動していれば問題ないはずです。
    embedding_model = OllamaEmbeddings(model=embedding_model_name) # Use dedicated embedding model

    # データを一時的なベクトルデータベース（Chroma）に保存
    vector_store = Chroma.from_documents(documents=sample_data, embedding=embedding_model)
    retriever = vector_store.as_retriever(search_kwargs={"k": 2})

    # プロンプトテンプレート
    prompt_template = """
    あなたは誠実で優秀な日本人のアシスタントです。
    必ず、以下の【提供された文脈】のみに基づいて質問に答えてください。
    文脈から判断できない場合は、「ドキュメント内に答えが見つかりませんでした」と答えてください。

    【提供された文脈】:
    {context}

    【質問】:
    {question}

    【回答】:
    """
    prompt = ChatPromptTemplate.from_template(prompt_template)
    llm = ChatOllama(model=chat_model_name, temperature=0) # Use generative chat model

    # チェーンの結合
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # 実行
    user_query = "プロジェクト・アルファのコードネームとリリース時期を教えて。"
    print(f"ユーザーの質問: {user_query}")
    print("GPUを活用して高速回答を生成中...")

    response = rag_chain.invoke(user_query)
    print(f"\n【ELYZAからの回答】:\n{response}")

# RAGの実行
run_colab_rag()

Ollamaをインストール中...
依存関係zstdをインストール中...
zstdのインストールに成功しました。
Ollamaインストールスクリプトをダウンロード中: https://ollama.com/install.sh
Ollamaインストールスクリプトを /tmp/install_ollama.sh にダウンロードしました。
Ollamaインストールスクリプトを実行中... (これには数分かかる場合があります)
インストールスクリプトの標準出力:

インストールスクリプトの標準エラー:
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Ollamaのインストールが完了しました。実行パス: /usr/local/bin/ollama
Ollamaサーバーを起動中...
Ollamaサーバーが起動しました。
Ollamaから lucas2024/llama-3-elyza-jp-8b:q5_k_m をダウンロード中... (約1〜2分)
モデルダウンロードの標準出力:

モデルダウンロードの標準エラー:
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 1bc7f8f7512b: 100% ▕█████████

## これまでのOllamaセットアップに関する課題とその解決策 (Issues and Solutions for Ollama Setup So Far)

このセクションでは、Ollamaのセットアップ中に直面した主な課題と、それらをどのように解決したかをまとめます。

### 1. `zstd` 依存関係の不足
*   **課題:** Ollamaのインストールスクリプトが、必要な圧縮ユーティリティ `zstd` がシステムにないために失敗しました。
*   **解決策:** `sudo apt-get install -y zstd` コマンドを実行して `zstd` を事前にインストールしました。

### 2. Ollamaインストールパスの不一致
*   **課題:** `OLLAMA_INSTALL_DIR` 環境変数を設定しても、Ollamaの公式インストールスクリプトが `/usr/local/bin` にOllamaをインストールするという独自のロジックを持っていました。これにより、Pythonスクリプトが誤ったパスのOllama実行ファイルを探索していました。
*   **解決策:** Pythonスクリプト内の `ollama_install_dir` および `ollama_executable_path` 変数を `/usr/local/bin` に明示的に設定し、スクリプトの動作に合わせて調整しました。

### 3. `--embeddings` フラグが認識されない
*   **課題:** 最初、LangChain `OllamaEmbeddings` から「This server does not support embeddings. Start it with --embeddings」というエラーメッセージを受け取ったため、`ollama serve` コマンドに `--embeddings` フラグを追加しようとしました。しかし、このフラグはOllamaの `serve` コマンドには存在しませんでした。
*   **解決策:** `ollama serve --help` の出力を確認し、`--embeddings` フラグが有効なオプションではないことを確認しました。`ollama serve` はフラグなしで実行し、モデルが埋め込み機能を提供することに依存するようにしました。

### 4. 埋め込み機能のモデル固有の要件
*   **課題:** `llama-3-elyza-jp-8b` モデルを `OllamaEmbeddings` に直接使用したところ、`ResponseError: This server does not support embeddings` というエラーが再度発生しました。これは、生成モデルが必ずしも埋め込み機能を提供しないことを示唆していました。
*   **解決策:** 生成タスクには `lucas2024/llama-3-elyza-jp-8b:q5_k_m` を引き続き使用しつつ、**専用の埋め込みモデル**である `nomic-embed-text` をOllamaから別途ダウンロードし、`OllamaEmbeddings` にそのモデルを使用するように変更しました。これにより、RAGパイプラインが正しく機能するようになりました。

これらの課題を一つずつ解決することで、Colab環境でOllamaを使用したRAGパイプラインを正常に構築することができました。